# HW4 — Olist Brazilian E-Commerce
**Реляційні бази даних · Google Colab · PostgreSQL**

Мета: завантажити датасет Olist, створити raw- і typed-таблиці, виконати DML, сегментацію покупців, JOIN-, агреговані, set-operation та window-запити.

**Перед запуском:**
1. Для `pgserver` використайте Python 3.12, якщо доступний.
2. У Colab Secrets додайте `KAGGLE_API_TOKEN` і дозвольте доступ ноутбуку.
3. Виконайте комірки зверху вниз.


## 1. Середовище та PostgreSQL

Потрібен Python 3.12 або нижче для сумісності з `pgserver`.


In [1]:
import sys
print("Python:", sys.version)
if sys.version_info >= (3, 13):
    raise RuntimeError(
        "Цей notebook використовує pgserver, який може бути недоступний для Python 3.13. "
        "У Colab відкрийте Command Palette (⌘+Shift+P) → Change runtime version "
        "і виберіть runtime із Python 3.12, якщо доступний; потім перезапустіть сесію."
    )

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [2]:
%pip uninstall -y -q kaggle kagglesdk
%pip install -q --no-cache-dir --upgrade kagglesdk
%pip install -q --no-cache-dir --upgrade kaggle pgserver psycopg2-binary sqlalchemy pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 14.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 210.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 203.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 205.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.6 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.6 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.6 which is

In [3]:
import os
import sys
import glob
import zipfile
import subprocess
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text


In [4]:
db = pgserver.get_server("/content/olist_pgdata")

from sqlalchemy.engine import make_url

db_url = make_url(db.get_uri()).set(drivername="postgresql+psycopg2")
engine = create_engine(db_url)

with engine.connect() as conn:
    print("PostgreSQL:", conn.execute(text("SELECT version()")).scalar())


PostgreSQL: PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## 2. Завантаження датасету

У Colab відкрийте **Secrets** (значок ключа) і додайте `KAGGLE_API_TOKEN` зі значенням нового токена Kaggle.
Увімкніть доступ до цього секрету для ноутбука.


In [5]:
from google.colab import userdata
import os
import json
import subprocess
import shutil

try:
    kaggle_token = userdata.get("KAGGLE_API_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Додай новий токен у Colab Secrets під назвою KAGGLE_API_TOKEN."
    ) from exc

os.environ["KAGGLE_API_TOKEN"] = kaggle_token

# Configure the legacy Kaggle CLI using the token-based access file.
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

access_token_path = os.path.join(kaggle_dir, "access_token")
with open(access_token_path, "w", encoding="utf-8") as f:
    f.write(kaggle_token)

os.chmod(access_token_path, 0o600)

download_dir = "/content/olist_data"
os.makedirs(download_dir, exist_ok=True)

kaggle_cli = shutil.which("kaggle")
if kaggle_cli is None:
    raise RuntimeError("Kaggle CLI не знайдено. Повторно виконай комірку встановлення бібліотек і перезапусти runtime.")

result = subprocess.run(
    [
        kaggle_cli, "datasets", "download",
        "-d", "olistbr/brazilian-ecommerce",
        "-p", download_dir,
        "--unzip",
    ],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(
        "Kaggle не завантажив датасет. Переконайся, що Kaggle CLI оновлений, "
        "а KAGGLE_API_TOKEN збережений у Colab Secrets. Деталі: "
        + (result.stderr or result.stdout)
    )

required_csvs = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
    "olist_order_reviews_raw": "olist_order_reviews_dataset.csv",
}

dataset_dir = None
for root, _, names in os.walk(download_dir):
    if set(required_csvs.values()).issubset(set(names)):
        dataset_dir = root
        break

if dataset_dir is None:
    raise FileNotFoundError(
        "У завантажених файлах не знайдено всі шість потрібних CSV."
    )

print("Dataset directory:", dataset_dir)


Dataset directory: /content/olist_data


Для цього завдання використовуються шість таблиць із курсової умови: customers, orders, order_items, products, sellers, order_reviews. Raw-таблиці завантажуються як текст, щоб порожні значення не спричиняли передчасного виведення типів.

In [6]:
files = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
    "olist_order_reviews_raw": "olist_order_reviews_dataset.csv",
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    if not os.path.exists(path):
        raise FileNotFoundError(f"У завантаженому датасеті не знайдено {file_name}")
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10_000)
    print(f"{table_name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

olist_customers_raw: 99,441 rows × 5 columns
olist_orders_raw: 99,441 rows × 8 columns
olist_order_items_raw: 112,650 rows × 7 columns
olist_products_raw: 32,951 rows × 9 columns
olist_sellers_raw: 3,095 rows × 4 columns
olist_order_reviews_raw: 99,224 rows × 7 columns


## 3. Typed-схема

Raw-таблиці зберігають вихідні CSV. У подальших запитах використовуються typed-таблиці з ключами, типами, constraints та індексами.

In [7]:
ddl_statements = [
    "DROP TABLE IF EXISTS customer_segments CASCADE",
    "DROP TABLE IF EXISTS seller_score CASCADE",
    "DROP TABLE IF EXISTS seller_alerts CASCADE",
    "DROP TABLE IF EXISTS olist_customer_dim CASCADE",
    "DROP TABLE IF EXISTS olist_order_reviews CASCADE",
    "DROP TABLE IF EXISTS olist_order_items CASCADE",
    "DROP TABLE IF EXISTS olist_orders CASCADE",
    "DROP TABLE IF EXISTS olist_customers CASCADE",
    "DROP TABLE IF EXISTS olist_products CASCADE",
    "DROP TABLE IF EXISTS olist_sellers CASCADE",

    """CREATE TABLE olist_customers (
        customer_id TEXT PRIMARY KEY,
        customer_unique_id TEXT NOT NULL,
        customer_zip_code_prefix INTEGER,
        customer_city TEXT,
        customer_state CHAR(2) NOT NULL CHECK (LENGTH(TRIM(customer_state)) = 2)
    )""",

    """CREATE TABLE olist_orders (
        order_id TEXT PRIMARY KEY,
        customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
        order_status TEXT NOT NULL CHECK (order_status IN
            ('created','approved','invoiced','processing','shipped','delivered','unavailable','canceled')),
        order_purchase_timestamp TIMESTAMP NOT NULL,
        order_approved_at TIMESTAMP,
        order_delivered_carrier_date TIMESTAMP,
        order_delivered_customer_date TIMESTAMP,
        order_estimated_delivery_date TIMESTAMP
    )""",

    """CREATE TABLE olist_products (
        product_id TEXT PRIMARY KEY,
        product_category_name TEXT,
        product_name_length INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
        product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
        product_photos_qty INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
        product_weight_g INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
        product_length_cm INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
        product_height_cm INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
        product_width_cm INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
    )""",

    """CREATE TABLE olist_sellers (
        seller_id TEXT PRIMARY KEY,
        seller_zip_code_prefix INTEGER,
        seller_city TEXT,
        seller_state CHAR(2) NOT NULL CHECK (LENGTH(TRIM(seller_state)) = 2)
    )""",

    """CREATE TABLE olist_order_items (
        order_id TEXT NOT NULL REFERENCES olist_orders(order_id),
        order_item_id INTEGER NOT NULL CHECK (order_item_id > 0),
        product_id TEXT REFERENCES olist_products(product_id),
        seller_id TEXT REFERENCES olist_sellers(seller_id),
        shipping_limit_date TIMESTAMP,
        price NUMERIC(12,2) CHECK (price IS NULL OR price >= 0),
        freight_value NUMERIC(12,2) CHECK (freight_value IS NULL OR freight_value >= 0),
        PRIMARY KEY (order_id, order_item_id)
    )""",

    """CREATE TABLE olist_order_reviews (
        review_row_id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
        review_id TEXT,
        order_id TEXT NOT NULL REFERENCES olist_orders(order_id),
        review_score SMALLINT CHECK (review_score BETWEEN 1 AND 5),
        review_comment_title TEXT,
        review_comment_message TEXT,
        review_creation_date TIMESTAMP,
        review_answer_timestamp TIMESTAMP
    )""",

    "CREATE INDEX idx_olist_customers_unique_id ON olist_customers(customer_unique_id)",
    "CREATE INDEX idx_olist_order_items_seller_order ON olist_order_items(seller_id, order_id)",
    "CREATE INDEX idx_olist_order_reviews_order_id ON olist_order_reviews(order_id)",
]

with engine.begin() as conn:
    for statement in ddl_statements:
        conn.exec_driver_sql(statement)

print("Typed tables and indexes created.")

Typed tables and indexes created.


### 3.1. Перенесення raw → typed

`NULLIF(column, '')` перетворює порожній рядок на SQL NULL до приведення типу. У CSV Olist назви двох полів продукту історично містять помилку `lenght`; у typed-схемі назви нормалізовано.

In [8]:
insert_statements = [
    """INSERT INTO olist_customers
       SELECT customer_id, customer_unique_id,
              NULLIF(customer_zip_code_prefix, '')::INTEGER,
              NULLIF(customer_city, ''), customer_state::CHAR(2)
       FROM olist_customers_raw""",

    """INSERT INTO olist_orders
       SELECT order_id, customer_id, order_status,
              NULLIF(order_purchase_timestamp, '')::TIMESTAMP,
              NULLIF(order_approved_at, '')::TIMESTAMP,
              NULLIF(order_delivered_carrier_date, '')::TIMESTAMP,
              NULLIF(order_delivered_customer_date, '')::TIMESTAMP,
              NULLIF(order_estimated_delivery_date, '')::TIMESTAMP
       FROM olist_orders_raw""",

    """INSERT INTO olist_products
       SELECT product_id, NULLIF(product_category_name, ''),
              NULLIF(product_name_lenght, '')::INTEGER,
              NULLIF(product_description_lenght, '')::INTEGER,
              NULLIF(product_photos_qty, '')::INTEGER,
              NULLIF(product_weight_g, '')::INTEGER,
              NULLIF(product_length_cm, '')::INTEGER,
              NULLIF(product_height_cm, '')::INTEGER,
              NULLIF(product_width_cm, '')::INTEGER
       FROM olist_products_raw""",

    """INSERT INTO olist_sellers
       SELECT seller_id, NULLIF(seller_zip_code_prefix, '')::INTEGER,
              NULLIF(seller_city, ''), seller_state::CHAR(2)
       FROM olist_sellers_raw""",

    """INSERT INTO olist_order_items
       SELECT order_id, order_item_id::INTEGER, NULLIF(product_id, ''),
              NULLIF(seller_id, ''), NULLIF(shipping_limit_date, '')::TIMESTAMP,
              NULLIF(price, '')::NUMERIC(12,2),
              NULLIF(freight_value, '')::NUMERIC(12,2)
       FROM olist_order_items_raw""",

    """INSERT INTO olist_order_reviews (
           review_id, order_id, review_score, review_comment_title,
           review_comment_message, review_creation_date, review_answer_timestamp
       )
       SELECT NULLIF(review_id, ''), order_id,
              NULLIF(review_score, '')::SMALLINT,
              NULLIF(review_comment_title, ''), NULLIF(review_comment_message, ''),
              NULLIF(review_creation_date, '')::TIMESTAMP,
              NULLIF(review_answer_timestamp, '')::TIMESTAMP
       FROM olist_order_reviews_raw""",
]

with engine.begin() as conn:
    for statement in insert_statements:
        conn.exec_driver_sql(statement)

print("Raw data copied into typed tables.")

Raw data copied into typed tables.


### 3.2. Smoke test: raw і typed row counts

In [9]:
count_sql = """
SELECT 'customers' AS entity,
       (SELECT COUNT(*) FROM olist_customers_raw) AS raw_rows,
       (SELECT COUNT(*) FROM olist_customers) AS typed_rows
UNION ALL
SELECT 'orders',
       (SELECT COUNT(*) FROM olist_orders_raw),
       (SELECT COUNT(*) FROM olist_orders)
UNION ALL
SELECT 'order_items',
       (SELECT COUNT(*) FROM olist_order_items_raw),
       (SELECT COUNT(*) FROM olist_order_items)
UNION ALL
SELECT 'products',
       (SELECT COUNT(*) FROM olist_products_raw),
       (SELECT COUNT(*) FROM olist_products)
UNION ALL
SELECT 'sellers',
       (SELECT COUNT(*) FROM olist_sellers_raw),
       (SELECT COUNT(*) FROM olist_sellers)
UNION ALL
SELECT 'reviews',
       (SELECT COUNT(*) FROM olist_order_reviews_raw),
       (SELECT COUNT(*) FROM olist_order_reviews)
ORDER BY entity
"""
pd.read_sql_query(text(count_sql), engine)

,entity,raw_rows,typed_rows
0,customers,99441,99441
1,order_items,112650,112650
2,orders,99441,99441
3,products,32951,32951
4,reviews,99224,99224
5,sellers,3095,3095


## 4. DML-запити з PostgreSQL-фічами

Кожен блок містить business-question, SQL і коротку інтерпретацію. Запити використовують constraints та `RETURNING` для контролю змін.

### 4.1. INSERT ... RETURNING — alert для продавця з найбільшою кількістю запізнень
**Business-question:** який продавець має найбільше доставлених із запізненням замовлень? Створімо alert для нього.

In [10]:
with engine.begin() as conn:
    conn.exec_driver_sql("""
        CREATE TABLE IF NOT EXISTS seller_alerts (
            alert_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            seller_id TEXT NOT NULL REFERENCES olist_sellers(seller_id),
            alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery','low_rating','data_quality')),
            severity SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
            details TEXT,
            created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
            UNIQUE (seller_id, alert_type)
        )
    """)
    alert_sql = """
    WITH seller_orders AS (
        SELECT DISTINCT seller_id, order_id
        FROM olist_order_items
        WHERE seller_id IS NOT NULL
    ),
    late_sellers AS (
        SELECT so.seller_id, COUNT(DISTINCT o.order_id) AS late_orders
        FROM seller_orders so
        JOIN olist_orders o ON o.order_id = so.order_id
        WHERE o.order_delivered_customer_date IS NOT NULL
          AND o.order_estimated_delivery_date IS NOT NULL
          AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
        GROUP BY so.seller_id
        ORDER BY late_orders DESC, so.seller_id
        LIMIT 1
    )
    INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
    SELECT seller_id, 'late_delivery', 4,
           'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
    FROM late_sellers
    ON CONFLICT (seller_id, alert_type) DO UPDATE
    SET severity = EXCLUDED.severity,
        details = EXCLUDED.details,
        created_at = NOW()
    RETURNING alert_id, seller_id, alert_type, severity, details, created_at
    """
    alert_df = pd.read_sql_query(text(alert_sql), conn)
display(alert_df)

,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the lo...,2026-10-17 01:58:20.975096+00:00


**Interpretation:** `RETURNING` показує рядок, який щойно створено або оновлено. `UNIQUE (seller_id, alert_type)` разом з `ON CONFLICT` не дозволяє повторним запускам накопичувати дублікати однакового alert.

### 4.2. INSERT ... ON CONFLICT DO UPDATE — seller score
**Business-question:** який середній review score та кількість оцінок пов’язані з кожним продавцем?

In [11]:
with engine.begin() as conn:
    conn.exec_driver_sql("""
        CREATE TABLE IF NOT EXISTS seller_score (
            seller_id TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
            avg_rating NUMERIC(3,2),
            n_reviews INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
            tier TEXT CHECK (tier IN ('gold','silver','bronze')),
            updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
        )
    """)
    seller_score_sql = """
    WITH seller_orders AS (
        SELECT DISTINCT seller_id, order_id
        FROM olist_order_items
        WHERE seller_id IS NOT NULL
    ),
    review_per_order AS (
        SELECT order_id, AVG(review_score)::NUMERIC(3,2) AS order_review_score
        FROM olist_order_reviews
        WHERE review_score IS NOT NULL
        GROUP BY order_id
    ),
    seller_rating AS (
        SELECT so.seller_id,
               AVG(rpo.order_review_score)::NUMERIC(3,2) AS avg_rating,
               COUNT(rpo.order_id)::INTEGER AS n_reviews
        FROM seller_orders so
        LEFT JOIN review_per_order rpo ON rpo.order_id = so.order_id
        GROUP BY so.seller_id
    )
    INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
    SELECT seller_id, avg_rating, n_reviews,
           CASE
             WHEN avg_rating IS NULL THEN NULL
             WHEN avg_rating >= 4.5 THEN 'gold'
             WHEN avg_rating >= 3.5 THEN 'silver'
             ELSE 'bronze'
           END
    FROM seller_rating
    ON CONFLICT (seller_id) DO UPDATE
    SET avg_rating = EXCLUDED.avg_rating,
        n_reviews = EXCLUDED.n_reviews,
        tier = EXCLUDED.tier,
        updated_at = NOW()
    RETURNING seller_id, avg_rating, n_reviews, tier, updated_at
    """
    score_df = pd.read_sql_query(text(seller_score_sql), conn)
display(score_df.head(10))

,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-10-17 01:58:21.218583+00:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-10-17 01:58:21.218583+00:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-10-17 01:58:21.218583+00:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-10-17 01:58:21.218583+00:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-10-17 01:58:21.218583+00:00
5,fc38b5dceee1a730fad8853453437fbd,4.60,5,gold,2026-10-17 01:58:21.218583+00:00
6,5def4c3732941a971cba8fdee992ede1,4.60,5,gold,2026-10-17 01:58:21.218583+00:00
7,b39d7fe263ef469605dbb32608aee0af,3.88,43,silver,2026-10-17 01:58:21.218583+00:00
8,ea65d8b58316a6f2362f2a9e4b3e86ad,4.00,9,silver,2026-10-17 01:58:21.218583+00:00
9,a45765f8afb1e594b22bf9e974b46765,4.00,1,silver,2026-10-17 01:58:21.218583+00:00


**Interpretation:** `seller_orders` задає гранулярність `seller_id × order_id`, а `review_per_order` зводить відгуки до одного рядка на замовлення, щоб позиції одного замовлення не множили review score. Відгук прив’язаний до замовлення, а не окремого продавця; у замовленнях із кількома продавцями це навчальне наближення. Продавець без оцінок може мати `avg_rating = NULL`, `n_reviews = 0`, `tier = NULL`.

### 4.3. UPDATE ... FROM — прапорець запізнення
**Business-question:** які замовлення доставлено пізніше за обіцяну дату?

In [12]:
with engine.begin() as conn:
    conn.exec_driver_sql(
        "ALTER TABLE olist_orders ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE"
    )
    update_late_sql = """
    WITH delivery_flags AS (
        SELECT order_id,
               (order_delivered_customer_date IS NOT NULL
                AND order_estimated_delivery_date IS NOT NULL
                AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE) AS is_late_calc
        FROM olist_orders
    )
    UPDATE olist_orders AS o
    SET is_late = f.is_late_calc
    FROM delivery_flags AS f
    WHERE f.order_id = o.order_id
    RETURNING o.order_id, o.customer_id, o.is_late
    """
    late_df = pd.read_sql_query(text(update_late_sql), conn)
display(late_df.head(10))

,order_id,customer_id,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,False
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,False
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,False
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,False
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,False


**Interpretation:** це ідемпотентне оновлення: кожен повторний запуск повторно обчислює `is_late` з дат, а не додає нові рядки.

### 4.4. DELETE ... RETURNING — очищення старих alert-ів

In [13]:
with engine.begin() as conn:
    deleted_alerts = pd.read_sql_query(text("""
        DELETE FROM seller_alerts
        WHERE created_at < NOW() - INTERVAL '30 days'
        RETURNING alert_id, seller_id, alert_type, created_at
    """), conn)
display(deleted_alerts)

,alert_id,seller_id,alert_type,created_at


**Interpretation:** якщо результат порожній, це нормально: у щойно створеній таблиці немає alert-ів старше 30 днів. `RETURNING` дає змогу аудитувати саме видалені рядки.

## 5. Customer dimension і сегментація покупців

Для customer-level метрик використовуємо `customer_unique_id`, бо один реальний покупець може мати кілька `customer_id`. Дату відліку беремо з самого датасету, а не з поточної дати, щоб результати були стабільними.

In [14]:
with engine.begin() as conn:
    conn.exec_driver_sql("DROP TABLE IF EXISTS customer_segments CASCADE")
    conn.exec_driver_sql("DROP TABLE IF EXISTS olist_customer_dim CASCADE")
    conn.exec_driver_sql("""
        CREATE TABLE olist_customer_dim (
            customer_unique_id TEXT PRIMARY KEY,
            latest_customer_id TEXT REFERENCES olist_customers(customer_id),
            customer_state CHAR(2),
            customer_city TEXT
        )
    """)
    conn.exec_driver_sql("""
        INSERT INTO olist_customer_dim
            (customer_unique_id, latest_customer_id, customer_state, customer_city)
        SELECT DISTINCT ON (c.customer_unique_id)
            c.customer_unique_id,
            c.customer_id,
            c.customer_state,
            c.customer_city
        FROM olist_customers c
        LEFT JOIN olist_orders o ON o.customer_id = c.customer_id
        ORDER BY c.customer_unique_id, o.order_purchase_timestamp DESC NULLS LAST, c.customer_id
    """)
    conn.exec_driver_sql("""
        CREATE TABLE customer_segments (
            segment_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            customer_unique_id TEXT NOT NULL UNIQUE
                REFERENCES olist_customer_dim(customer_unique_id) ON DELETE CASCADE,
            segment_name TEXT NOT NULL CHECK (segment_name IN ('VIP','regular','new','inactive')),
            rfm_score SMALLINT NOT NULL CHECK (rfm_score BETWEEN 1 AND 5),
            recency_days INTEGER CHECK (recency_days IS NULL OR recency_days >= 0),
            monetary_value NUMERIC(12,2) NOT NULL DEFAULT 0 CHECK (monetary_value >= 0),
            n_orders INTEGER NOT NULL DEFAULT 0 CHECK (n_orders >= 0),
            assigned_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
        )
    """)
print("Customer dimension and segment table created.")

Customer dimension and segment table created.


In [15]:
segment_sql = """
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT o.order_id, o.customer_id, o.order_purchase_timestamp::DATE AS order_date,
           COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12,2) AS order_total
    FROM olist_orders o
    LEFT JOIN olist_order_items oi ON oi.order_id = o.order_id
    GROUP BY o.order_id, o.customer_id, o.order_purchase_timestamp
),
customer_stats AS (
    SELECT c.customer_unique_id,
           COUNT(DISTINCT ot.order_id)::INTEGER AS n_orders,
           COALESCE(SUM(ot.order_total), 0)::NUMERIC(12,2) AS monetary_value,
           MAX(ot.order_date) AS last_order_date
    FROM olist_customers c
    LEFT JOIN order_totals ot ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT cs.customer_unique_id, cs.n_orders, cs.monetary_value,
           CASE WHEN cs.last_order_date IS NULL THEN NULL
                ELSE (rd.as_of_date - cs.last_order_date)::INTEGER END AS recency_days,
           CASE
             WHEN cs.monetary_value >= 1000 THEN 'VIP'
             WHEN cs.last_order_date IS NOT NULL
                  AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                  AND cs.n_orders = 1 THEN 'new'
             WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
             ELSE 'inactive'
           END AS segment_name,
           CASE
             WHEN cs.monetary_value >= 1000 THEN 5
             WHEN cs.monetary_value >= 500 THEN 4
             WHEN cs.n_orders >= 2 THEN 3
             WHEN cs.n_orders = 1 THEN 2
             ELSE 1
           END::SMALLINT AS rfm_score
    FROM customer_stats cs
    CROSS JOIN reference_date rd
)
INSERT INTO customer_segments
    (customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders)
SELECT customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE
SET segment_name = EXCLUDED.segment_name,
    rfm_score = EXCLUDED.rfm_score,
    recency_days = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders = EXCLUDED.n_orders,
    assigned_at = NOW()
RETURNING customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders
"""
with engine.begin() as conn:
    segments_df = pd.read_sql_query(text(segment_sql), conn)
display(segments_df.head(10))

,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1
5,0004bd2a26a76fe21f786e4fbd80607f,regular,2,196,166.98,1
6,00050ab1314c0e55a6ca13cf7181fecf,inactive,2,181,35.38,1
7,00053a61a98854899e70ed204dd4bafe,regular,2,232,419.18,1
8,0005e1862207bf6ccc02e4228effd9a0,regular,2,593,150.12,1
9,0005ef4cd20d2893f0d9fbd94d3c0d97,regular,2,220,129.76,1


**Пояснення:** `olist_customer_dim` містить один рядок на `customer_unique_id` і дозволяє визначити FK. `reference_date` залежить від максимального timestamp у датасеті, тому сегменти не змінюються лише через те, що notebook запущено в інший день. `ON CONFLICT` оновлює сегмент наявного покупця замість створення дубліката. Пороги сегментації — навчальні правила, а не офіційна бізнес-класифікація Olist.

### 5.1. Перевірка constraints

In [16]:
constraints_sql = """
SELECT conname, contype, pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname
"""
pd.read_sql_query(text(constraints_sql), engine)

,conname,contype,definition
0,customer_segments_customer_unique_id_fkey,f,FOREIGN KEY (customer_unique_id) REFERENCES ol...
1,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
2,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
3,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
4,customer_segments_pkey,p,PRIMARY KEY (segment_id)
5,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_day...
6,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
7,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text,..."


## 6. JOIN-запити з business interpretation

Кожен запит повертає фактичний output; короткі інтерпретації наведені після нього.

### 6.1. INNER JOIN — категорії та штати продавців із найбільшим revenue
**Business-question:** які категорії та штати продавців дають найбільший дохід у delivered-замовленнях?

In [17]:
q_join_inner = """
SELECT p.product_category_name, s.seller_state,
       COUNT(DISTINCT oi.order_id) AS n_orders,
       SUM(oi.price + oi.freight_value)::NUMERIC(12,2) AS revenue
FROM olist_order_items oi
JOIN olist_orders o ON o.order_id = oi.order_id
JOIN olist_products p ON p.product_id = oi.product_id
JOIN olist_sellers s ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10
"""
pd.read_sql_query(text(q_join_inner), engine)

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


**Interpretation:** результат ранжує поєднання категорії та штату за сумою вартості товарів і доставки в доставлених замовленнях. Це не чиста маржа: витрати продавця тут не враховані.

### 6.2. LEFT JOIN + COALESCE — customer features
**Business-question:** які customer-level features можна сформувати для кожного реального покупця, включно з тими, у кого немає метрик замовлень?

In [18]:
q_join_left = """
WITH order_totals AS (
    SELECT o.order_id, c.customer_unique_id,
           COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12,2) AS order_total
    FROM olist_orders o
    JOIN olist_customers c ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items oi ON oi.order_id = o.order_id
    GROUP BY o.order_id, c.customer_unique_id
),
review_per_order AS (
    SELECT order_id, AVG(review_score)::NUMERIC(3,2) AS order_review_score
    FROM olist_order_reviews
    WHERE review_score IS NOT NULL
    GROUP BY order_id
),
customer_stats AS (
    SELECT ot.customer_unique_id,
           COUNT(DISTINCT ot.order_id) AS n_orders,
           COALESCE(SUM(ot.order_total), 0)::NUMERIC(12,2) AS total_spend,
           AVG(rpo.order_review_score)::NUMERIC(3,2) AS avg_review_score
    FROM order_totals ot
    LEFT JOIN review_per_order rpo ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT cd.customer_unique_id, cd.customer_state,
       COALESCE(cs.n_orders, 0) AS n_orders,
       COALESCE(cs.total_spend, 0)::NUMERIC(12,2) AS total_spend,
       COALESCE(cs.avg_review_score, 0)::NUMERIC(3,2) AS avg_review_score
FROM olist_customer_dim cd
LEFT JOIN customer_stats cs ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 20
"""
pd.read_sql_query(text(q_join_left), engine)

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.0
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.0
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.0
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.0
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.0
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.0
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.0
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.0
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.0
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.0


**Interpretation:** `LEFT JOIN` зберігає всіх покупців із dimension, а `COALESCE` заповнює відсутні числові метрики нулями. `review_per_order` не дозволяє множити оцінки через кілька товарних позицій. Нуль для відсутнього review — це технічна домовленість для демонстрації; у ML краще зберігати окремий missingness indicator.

### 6.3. FULL OUTER JOIN — порівняння штатів клієнтів і продавців
**Business-question:** які штати є лише в клієнтів, лише в продавців або в обох наборах?

In [19]:
q_join_full = """
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers
    FROM olist_customers GROUP BY customer_state
),
seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers GROUP BY seller_state
)
SELECT COALESCE(c.state, s.state) AS state,
       c.n_customers, s.n_sellers,
       CASE WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
            WHEN c.state IS NOT NULL THEN 'customers_only'
            ELSE 'sellers_only' END AS state_presence
FROM customer_states c
FULL OUTER JOIN seller_states s ON s.state = c.state
ORDER BY state
"""
pd.read_sql_query(text(q_join_full), engine)

,state,n_customers,n_sellers,state_presence
0,AC,81,1.0,both
1,AL,413,NaN,customers_only
2,AM,148,1.0,both
3,AP,68,NaN,customers_only
4,BA,3380,19.0,both
5,CE,1336,13.0,both
6,DF,2140,30.0,both
7,ES,2033,23.0,both
8,GO,2020,40.0,both
9,MA,747,1.0,both


**Interpretation:** `FULL OUTER JOIN` зберігає всі штати з обох наборів і дозволяє побачити асиметрію географічного покриття.

### 6.4. SELF JOIN — пари продавців з одного штату
**Business-question:** які пари продавців працюють у тому самому штаті, але в різних містах?

In [20]:
q_join_self = """
SELECT s1.seller_id AS seller_a, s2.seller_id AS seller_b,
       s1.seller_state, s1.seller_city AS city_a, s2.seller_city AS city_b
FROM olist_sellers s1
JOIN olist_sellers s2
  ON s1.seller_state = s2.seller_state
 AND s1.seller_id < s2.seller_id
 AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state, seller_a, seller_b
LIMIT 20
"""
pd.read_sql_query(text(q_join_self), engine)

,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


**Interpretation:** умова `seller_id < seller_id` усуває дзеркальні дублікати та пари продавця із самим собою. Це лише кандидати для регіонального порівняння, а не доказ конкурентної взаємодії.

### 6.5. Anti-join — замовлення без відгуку
**Business-question:** які замовлення не мають жодного review-запису?

In [21]:
q_join_anti = """
SELECT o.order_id, o.customer_id, o.order_status, o.order_purchase_timestamp
FROM olist_orders o
LEFT JOIN olist_order_reviews rv ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
ORDER BY o.order_purchase_timestamp
LIMIT 20
"""
pd.read_sql_query(text(q_join_anti), engine)

,order_id,customer_id,order_status,order_purchase_timestamp
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
5,d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41
6,05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28
7,719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51
8,fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36
9,171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37


**Interpretation:** anti-join знаходить замовлення без відповідного рядка у reviews. Відсутність review не означає автоматично негативний досвід — клієнт міг не залишити відгук.

## 7. GROUP BY, HAVING та агрегати

`WHERE` відбирає рядки до агрегації, а `HAVING` — уже сформовані групи.

### 7.1. Revenue за категоріями
**Business-question:** які категорії мають сумарну вартість товарів понад 10 000?

In [22]:
q_agg1 = """
SELECT p.product_category_name,
       COUNT(DISTINCT oi.order_id) AS n_orders,
       SUM(oi.price)::NUMERIC(12,2) AS gross_revenue,
       SUM(oi.freight_value)::NUMERIC(12,2) AS freight,
       AVG(oi.price)::NUMERIC(8,2) AS avg_item_price
FROM olist_order_items oi
JOIN olist_products p ON p.product_id = oi.product_id
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 10000
ORDER BY gross_revenue DESC NULLS LAST
"""
pd.read_sql_query(text(q_agg1), engine)

,product_category_name,n_orders,gross_revenue,freight,avg_item_price
0,beleza_saude,8836,1258681.34,182566.73,130.16
1,relogios_presentes,5624,1205005.68,100535.93,201.14
2,cama_mesa_banho,9417,1036988.68,204693.04,93.30
3,esporte_lazer,7720,988048.97,168607.51,114.34
4,informatica_acessorios,6689,911954.32,147318.08,116.51
5,moveis_decoracao,6449,729762.49,172749.30,87.56
6,cool_stuff,3632,635290.85,84039.10,167.36
7,utilidades_domesticas,5884,632248.66,146149.11,90.79
8,automotivo,3897,592720.11,92664.21,139.96
9,ferramentas_jardim,3518,485256.46,98962.75,111.63


**Interpretation:** `HAVING` залишає категорії, які перетнули поріг revenue; `WHERE` не можна застосувати до `SUM()` на рівні окремого рядка до групування.

### 7.2. AOV за штатом клієнта
**Business-question:** які штати мають найбільший average order value серед штатів із щонайменше 100 замовленнями?

In [23]:
q_agg2 = """
WITH order_totals AS (
    SELECT o.order_id, c.customer_state,
           SUM(oi.price + oi.freight_value)::NUMERIC(12,2) AS order_total
    FROM olist_orders o
    JOIN olist_customers c ON c.customer_id = o.customer_id
    JOIN olist_order_items oi ON oi.order_id = o.order_id
    GROUP BY o.order_id, c.customer_state
)
SELECT customer_state, COUNT(*) AS n_orders,
       SUM(order_total)::NUMERIC(12,2) AS gmv,
       (SUM(order_total) / NULLIF(COUNT(*),0))::NUMERIC(8,2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY aov DESC NULLS LAST
"""
pd.read_sql_query(text(q_agg2), engine)

,customer_state,n_orders,gmv,aov
0,PB,532,140987.81,265.01
1,AL,411,96229.40,234.13
2,RO,247,57558.02,233.03
3,PA,970,217647.11,224.38
4,TO,279,61354.42,219.91
5,PI,493,108132.28,219.34
6,SE,345,73032.32,211.69
7,RN,482,101895.08,211.40
8,CE,1327,275606.30,207.69
9,MT,903,186168.96,206.17


**Interpretation:** AOV — середня сума замовлення з доставкою. Поріг `HAVING COUNT(*) >= 100` зменшує вплив штатів із дуже малою кількістю спостережень.

### 7.3. Категорії, які купував реальний покупець через STRING_AGG
**Business-question:** які категорії зустрічаються в історії покупок клієнтів із найбільшою сумою витрат?

In [24]:
q_agg3 = """
SELECT c.customer_unique_id,
       COUNT(DISTINCT o.order_id) AS n_orders,
       SUM(oi.price)::NUMERIC(12,2) AS total_spend,
       STRING_AGG(
           DISTINCT COALESCE(p.product_category_name, '(unknown)'),
           ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
       ) AS categories
FROM olist_customers c
JOIN olist_orders o ON o.customer_id = c.customer_id
JOIN olist_order_items oi ON oi.order_id = o.order_id
LEFT JOIN olist_products p ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 1
ORDER BY total_spend DESC NULLS LAST
LIMIT 20
"""
pd.read_sql_query(text(q_agg3), engine)

,customer_unique_id,n_orders,total_spend,categories
0,0a0a92112bd4c708ca5fde585afaa872,1,13440.00,telefonia_fixa
1,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7160.00,telefonia_fixa
3,dc4802a71eae9be1dd28f5d788ceb526,1,6735.00,utilidades_domesticas
4,459bef486812aa25204be022145caa62,1,6729.00,pcs
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.00,artes
6,4007669dec559734d6f53e029e360987,1,5934.60,agro_industria_e_comercio
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.00,eletroportateis
8,5d0a2980b292d049061542014e8960bf,1,4599.90,cool_stuff
9,48e1ac109decbb87765a3eade6854098,1,4590.00,pcs


**Interpretation:** `STRING_AGG(DISTINCT ...)` формує компактний список категорій покупця без повторення однієї категорії в межах рядка. Тут `total_spend` рахується за ціною товарів без доставки.

## 8. Set operations і window function

### 8.1. UNION — штати, присутні серед клієнтів або продавців
**Business-question:** які коди штатів присутні в обох ролях?

In [25]:
q_union = """
SELECT customer_state AS state, 'customer' AS role FROM olist_customers
UNION
SELECT seller_state AS state, 'seller' AS role FROM olist_sellers
ORDER BY state, role
"""
pd.read_sql_query(text(q_union), engine)

,state,role
0,AC,customer
1,AC,seller
2,AL,customer
3,AM,customer
4,AM,seller
5,AP,customer
6,BA,customer
7,BA,seller
8,CE,customer
9,CE,seller


**Interpretation:** `UNION` усуває дублікати однакових пар `(state, role)`. Оскільки роль входить до результату, один штат може з'явитися двічі — окремо для клієнтів і продавців.

### 8.2. INTERSECT — штати, де є клієнти та продавці

In [26]:
q_intersect = """
SELECT customer_state AS state FROM olist_customers
INTERSECT
SELECT seller_state AS state FROM olist_sellers
ORDER BY state
"""
pd.read_sql_query(text(q_intersect), engine)

,state
0,AC
1,AM
2,BA
3,CE
4,DF
5,ES
6,GO
7,MA
8,MG
9,MS


**Interpretation:** `INTERSECT` повертає лише спільні коди штатів і автоматично прибирає дублікати.

### 8.3. ROW_NUMBER — найновіше замовлення кожного реального покупця
**Business-question:** яке останнє за часом замовлення кожного `customer_unique_id`?

In [27]:
q_window = """
WITH ranked_orders AS (
    SELECT c.customer_unique_id, o.customer_id, o.order_id, o.order_purchase_timestamp,
           ROW_NUMBER() OVER (
               PARTITION BY c.customer_unique_id
               ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
           ) AS rn
    FROM olist_orders o
    JOIN olist_customers c ON c.customer_id = o.customer_id
)
SELECT customer_unique_id, customer_id, order_id, order_purchase_timestamp
FROM ranked_orders
WHERE rn = 1
ORDER BY customer_unique_id
LIMIT 20
"""
pd.read_sql_query(text(q_window), engine)

,customer_unique_id,customer_id,order_id,order_purchase_timestamp
0,0000366f3b9a7992bf8c76cfdf3221e2,fadbb3709178fc513abc1b2670aa1ad2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4cb282e167ae9234755102258dd52ee8,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,9b3932a6253894a02c1df9d19004239f,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,914991f0c02ef0843c0e7010c819d642,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,47227568b10f5f58a524a75507e6992c,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42
5,0004bd2a26a76fe21f786e4fbd80607f,4a913a170c26e3c8052ed0202849b5a8,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16
6,00050ab1314c0e55a6ca13cf7181fecf,d2509c13692836fc0449e88cf9eb4858,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23
7,00053a61a98854899e70ed204dd4bafe,a81ebb9b32f102298c0c89635b4b3154,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41
8,0005e1862207bf6ccc02e4228effd9a0,3b37fb626fdf46cd99d37ec62afa88ff,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12
9,0005ef4cd20d2893f0d9fbd94d3c0d97,c59e8ff99836e90d8b457d4122dc34e9,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12


**Interpretation:** `ROW_NUMBER()` нумерує замовлення всередині кожного реального покупця від найновішого до найстарішого. Додатковий порядок за `order_id` забезпечує детермінований tie-break, коли timestamps однакові.

## 9. Підсумкові перевірки та короткий аудит

In [28]:
audit_sql = """
SELECT 'orders_with_late_flag' AS metric, COUNT(*)::BIGINT AS value
FROM olist_orders WHERE is_late
UNION ALL
SELECT 'orders_without_review', COUNT(*)::BIGINT
FROM olist_orders o
WHERE NOT EXISTS (
    SELECT 1 FROM olist_order_reviews rv WHERE rv.order_id = o.order_id
)
UNION ALL
SELECT 'customers_in_segments', COUNT(*)::BIGINT FROM customer_segments
UNION ALL
SELECT 'sellers_with_score', COUNT(*)::BIGINT FROM seller_score
ORDER BY metric
"""
pd.read_sql_query(text(audit_sql), engine)

,metric,value
0,customers_in_segments,96096
1,orders_with_late_flag,6535
2,orders_without_review,768
3,sellers_with_score,3095


## 10. Reflection (200–250 слів)

Найскладнішим для мене був `FULL OUTER JOIN`, оскільки він має зберігати рядки з обох наборів, навіть якщо відповідного штату немає з іншого боку. Для коректної інтерпретації довелося явно розрізнити `both`, `customers_only` та `sellers_only`. Найскладніше бізнес-питання — оцінювання продавців за відгуками: review у Olist прив’язаний до замовлення, а одне замовлення може містити товари кількох продавців.

Для production-ETL найбільш корисною вважаю комбінацію `ON CONFLICT` і constraints. Constraints не дозволяють записувати недопустимі значення, а upsert дає змогу безпечно перераховувати метрики без дублювання рядків. `RETURNING` також важливий для аудиту змін. Identity-ключі спрощують ідентифікацію записів, а window functions допомагають вибирати останню подію, ранжувати товари або будувати часові ознаки.

У реальному ML-проєкті window functions використала б для останнього замовлення клієнта, кількості покупок за попередні 30/90 днів та послідовності подій. Серед проблем якості даних очікую відсутні відгуки, порожні назви категорій, порожні тексти коментарів і неповні дати доставки. Відсутність review не слід трактувати як низьку оцінку.

Для customer-level ML-фіч використовується `customer_unique_id`, бо один реальний покупець може мати кілька `customer_id`. Інакше повторні покупки були б розділені, а frequency штучно занижена. Перед обчисленням `seller_score` дані приводяться до гранулярності `seller_id × order_id`, щоб кілька позицій одного замовлення не множили вагу review. Оскільки один order-level review може бути пов’язаний із кількома продавцями, цю ознаку потрібно трактувати як наближення, а не точну індивідуальну оцінку продавця.

## Джерело

Olist Brazilian E-Commerce Public Dataset: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
